# Level 2: final development training

Compare the same three configurations using this level's development folds, then freeze the lowest-score configuration and the median of its five best validation epochs. The score is the mean pooled simulation RMSE ratio relative to the baseline across both targets.

Train a fresh model on **all 266 development simulations (3,990 maps)** with new scalers. This notebook never loads test data. The two later levels' comparisons and selections are completed before either test evaluation.

Select `.venv/bin/python` and restart the kernel after module edits. Existing checkpoints reload without retraining. A frozen selection must match its recorded comparison; intentional new experiments need separate selection/checkpoint files. The 20-epoch development limit and single seed remain limitations; the selected duration is a development-based heuristic, not a guaranteed optimum.


## 1. Locate the project and import shared helpers

Keep level-specific outputs together and reuse the training and checkpoint functions.

In [1]:
from pathlib import Path
from hashlib import sha256
import sys
import json

PROJECT_ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/"mlp"/"data.py").is_file()),None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook inside the cosmoreg project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

import numpy as np
import torch
from mlp import data
from mlp.final_train import train_final_model,save_final_model,load_final_model

LEVEL=2
LEVEL_ROOT=PROJECT_ROOT/"mlp"/f"level{LEVEL}"
SPECTRA_ROOT=PROJECT_ROOT/"powerspectrum"
COMPARISON_PATH=LEVEL_ROOT/f"level{LEVEL}_config_comparison.json"
SELECTION_PATH=LEVEL_ROOT/f"level{LEVEL}_final_selection.json"
print("Project:",PROJECT_ROOT)


Project: /home/meatpigeon/Personal/classes/csci494/cosmoreg


## 2. Choose and freeze using development results only

Selection uses the same predefined score as level 1. All optimizer/scaling choices are inherited from the winning development configuration. Final epochs are the median of that configuration's best epochs. Save this choice before final fitting or test access; the comparison hash records its source.

In [2]:
comparison=json.loads(COMPARISON_PATH.read_text())
assert comparison["level"]==LEVEL
assert len(comparison["ranking"])==3
chosen_name=comparison["ranking"][0]
assert chosen_name==min(comparison["summaries"],key=lambda name:comparison["summaries"][name]["score"])
best_epochs=comparison["summaries"][chosen_name]["best_epochs"]
assert len(best_epochs)==5
FINAL_SETTINGS=dict(comparison["configs"][chosen_name])
FINAL_SETTINGS["epochs"]=int(np.median(best_epochs))
selection={
    "level":LEVEL,
    "configuration_name":chosen_name,
    "settings":FINAL_SETTINGS,
    "development_best_epochs":best_epochs,
    "epoch_selection":"median best epoch from five development folds",
    "selection_criterion":comparison["selection_criterion"],
    "comparison_sha256":sha256(COMPARISON_PATH.read_bytes()).hexdigest(),
}
if SELECTION_PATH.exists():
    assert json.loads(SELECTION_PATH.read_text())==selection,"Existing frozen selection differs."
else:
    with SELECTION_PATH.open("x") as file:
        json.dump(selection,file,indent=2,allow_nan=False)
        file.write("\n")
CHECKPOINT_PATH=LEVEL_ROOT/"checkpoints"/f"level{LEVEL}_{chosen_name}_seed{FINAL_SETTINGS['seed']}_epochs{FINAL_SETTINGS['epochs']}.pt"
print("Selected configuration:",chosen_name)
print("Architecture:",[50,*FINAL_SETTINGS["hidden_dims"],2])
print("Weight decay:",FINAL_SETTINGS["weight_decay"])
print("Best validation epochs:",best_epochs)
print("Fixed final duration:",FINAL_SETTINGS["epochs"])
print("Frozen selection:",SELECTION_PATH)


Selected configuration: stronger_decay
Architecture: [50, 32, 16, 2]
Weight decay: 0.01
Best validation epochs: [13, 13, 19, 11, 13]
Fixed final duration: 13
Frozen selection: /home/meatpigeon/Personal/classes/csci494/cosmoreg/mlp/level2/level2_final_selection.json


## 3. Train on all development folds and save

Use a fresh model, optimizer, and development-fitted scalers. Keep the fixed final epoch's weights. Reported losses are batch-weighted training MSE, not validation/test performance. Checkpoints are gitignored and overwrite-protected.

In [3]:
if CHECKPOINT_PATH.exists():
    final_results=load_final_model(CHECKPOINT_PATH)
    print("Reloaded:",CHECKPOINT_PATH)
else:
    previous_threads=torch.get_num_threads()
    torch.set_num_threads(1)
    try:
        final_results=train_final_model(SPECTRA_ROOT,level=LEVEL,**FINAL_SETTINGS)
    finally:
        torch.set_num_threads(previous_threads)
    final_results["settings"]["configuration_name"]=chosen_name
    final_results["settings"]["epoch_selection"]=selection["epoch_selection"]
    final_results["settings"]["development_best_epochs"]=best_epochs
    final_results["settings"]["comparison_sha256"]=selection["comparison_sha256"]
    save_final_model(final_results,CHECKPOINT_PATH)
    print("Saved:",CHECKPOINT_PATH)
assert final_results["settings"]["level"]==LEVEL
for key,value in FINAL_SETTINGS.items():
    assert final_results["settings"][key]==value
assert final_results["settings"]["configuration_name"]==chosen_name
assert final_results["settings"]["comparison_sha256"]==selection["comparison_sha256"]
assert final_results["training_maps"]==3990
assert len(final_results["training_simulation_ids"])==266
assert len(final_results["train_history"])==FINAL_SETTINGS["epochs"]
print("Development maps:",final_results["training_maps"])
print("Development simulations:",len(final_results["training_simulation_ids"]))


Epoch 01/13 | Training MSE: 0.7598
Epoch 02/13 | Training MSE: 0.6780
Epoch 03/13 | Training MSE: 0.6572
Epoch 04/13 | Training MSE: 0.6370
Epoch 05/13 | Training MSE: 0.6285
Epoch 06/13 | Training MSE: 0.6200
Epoch 07/13 | Training MSE: 0.6132
Epoch 08/13 | Training MSE: 0.6133
Epoch 09/13 | Training MSE: 0.6076
Epoch 10/13 | Training MSE: 0.6072
Epoch 11/13 | Training MSE: 0.6047
Epoch 12/13 | Training MSE: 0.5992
Epoch 13/13 | Training MSE: 0.6008
Saved: /home/meatpigeon/Personal/classes/csci494/cosmoreg/mlp/level2/checkpoints/level2_stronger_decay_seed42_epochs13.pt
Development maps: 3990
Development simulations: 266


## 4. Verify saved scalers and prediction roundtrip

Use development data to check exact save/load agreement and verify that scalers were fitted on the entire development set. These are serialization checks, not held-out performance estimates. Test arrays remain unopened.

In [4]:
loaded_results=load_final_model(CHECKPOINT_PATH)
X_dev,y_dev,ids_dev=data.load_development(SPECTRA_ROOT,LEVEL)
s=loaded_results["scalers"]
feature_mean,feature_std=data.fit_feature_scaler(X_dev)
target_mean,target_std=data.fit_target_scaler(y_dev)
for name,expected in (("feature_mean",feature_mean),("feature_std",feature_std),("target_mean",target_mean),("target_std",target_std)):
    assert np.array_equal(s[name],expected)
assert loaded_results["training_simulation_ids"]==np.unique(ids_dev).tolist()
X_tensor=torch.from_numpy(data.transform_features(X_dev,s["feature_mean"],s["feature_std"]))
with torch.no_grad():
    before_save=final_results["model"](X_tensor)
    after_load=loaded_results["model"](X_tensor)
assert torch.equal(before_save,after_load)
assert after_load.shape==(3990,2)
assert torch.isfinite(after_load).all()
print("PASS: complete-development scalers and checkpoint predictions verified.")
print("Test data remains untouched.")


PASS: complete-development scalers and checkpoint predictions verified.
Test data remains untouched.
